# 04 — Model Development and Tuning

Run from the repository root after `python -m src.utils.prepare_data`. This notebook compares Logistic Regression, Random Forest and XGBoost using the prepared splits. It chooses a probability threshold on validation data and reports test metrics only after selection. The script `src/models/train.py` is the canonical reproducible training entry point.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, ConfusionMatrixDisplay, PrecisionRecallDisplay)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from src.features.build_features import build_preprocessor
ROOT=Path.cwd()
DATA=ROOT/"data"/"processed"
def load_split(name):
    df=pd.read_csv(DATA/f"{name}.csv")
    y=df.pop("churn").astype(int)
    return df,y
X_train,y_train=load_split("train")
X_valid,y_valid=load_split("valid")
X_test,y_test=load_split("test")
print({"train":X_train.shape,"valid":X_valid.shape,"test":X_test.shape})

## Candidate models and validation comparison

The search below is intentionally bounded so it can run on a student laptop. If runtime is constrained, use the already prepared `src/models/train.py` pipeline and report its saved comparison CSV. MLflow tracking is optional at runtime; see the requirements document for the experiment fields that should be logged.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import ParameterSampler
pre=build_preprocessor(X_train)
models={"logistic_regression":LogisticRegression(max_iter=1500,class_weight="balanced",random_state=42),"random_forest":RandomForestClassifier(n_estimators=250,min_samples_leaf=2,class_weight="balanced",random_state=42,n_jobs=-1)}
try:
    from xgboost import XGBClassifier
    neg=int((y_train==0).sum()); pos=max(1,int((y_train==1).sum()))
    models["xgboost"]=XGBClassifier(n_estimators=300,max_depth=4,learning_rate=.04,subsample=.9,colsample_bytree=.9,objective="binary:logistic",eval_metric="logloss",scale_pos_weight=neg/pos,random_state=42,n_jobs=-1)
except ImportError: print("XGBoost not installed; skipping it")
def threshold_for_f1(y,p):
    from sklearn.metrics import precision_recall_curve
    pr,re,th=precision_recall_curve(y,p)
    if not len(th): return .5
    f=(2*pr[:-1]*re[:-1])/(pr[:-1]+re[:-1]+1e-12)
    return float(th[int(np.nanargmax(f))])
rows=[]; fitted={}
for name,est in models.items():
    pipe=Pipeline([("pre",clone(pre)),("model",est)])
    pipe.fit(X_train,y_train); p=pipe.predict_proba(X_valid)[:,1]; t=threshold_for_f1(y_valid,p); pred=(p>=t).astype(int)
    rows.append({"model":name,"threshold":t,"roc_auc":roc_auc_score(y_valid,p),"pr_auc":average_precision_score(y_valid,p),"f1":f1_score(y_valid,pred,zero_division=0),"precision":precision_score(y_valid,pred,zero_division=0),"recall":recall_score(y_valid,pred,zero_division=0)})
    fitted[name]=pipe
comparison=pd.DataFrame(rows).sort_values(["f1","pr_auc"],ascending=False)
comparison.to_csv(ROOT/"reports"/"notebook_model_comparison.csv",index=False)
display(comparison)

## Bounded hyperparameter search (optional, more compute)

Run this after the first comparison if time allows. It tunes Random Forest settings using validation-oriented cross-validation on the training split. For strict final evaluation, keep the test split untouched until after model and threshold selection. MLflow can be used to record each candidate and its metrics.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint, uniform
rf_pipe=Pipeline([("pre",clone(pre)),("model",RandomForestClassifier(class_weight="balanced",random_state=42,n_jobs=-1))])
param_dist={"model__n_estimators": randint(150,401), "model__max_depth": [None, 6, 10, 14], "model__min_samples_leaf": randint(1,5), "model__max_features": ["sqrt", 0.7]}
search=RandomizedSearchCV(rf_pipe,param_distributions=param_dist,n_iter=8,scoring="average_precision",cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42),random_state=42,n_jobs=-1,verbose=1,refit=True)
search.fit(X_train,y_train)
tuned_p=search.best_estimator_.predict_proba(X_valid)[:,1]
print("Best parameters:",search.best_params_)
print("CV average precision:",search.best_score_)
print("Validation PR-AUC:",average_precision_score(y_valid,tuned_p))
pd.DataFrame(search.cv_results_).sort_values("rank_test_score").to_csv(ROOT/"reports"/"random_forest_search_results.csv",index=False)

## Threshold and held-out test evaluation

Select the model based on a stated validation objective. Here the example uses validation F1 as the primary ranking metric; a business may instead prioritize recall subject to a minimum precision or contact capacity. Do not repeatedly tune against test data.

In [ ]:
winner_name=str(comparison.iloc[0]["model"]); winner=fitted[winner_name]; threshold=float(comparison.iloc[0]["threshold"])
X_fit=pd.concat([X_train,X_valid],ignore_index=True); y_fit=pd.concat([y_train,y_valid],ignore_index=True)
# Refit the selected estimator on train + validation after model/threshold selection.
from sklearn.base import clone
final_model=clone(winner); final_model.fit(X_fit,y_fit)
p_test=final_model.predict_proba(X_test)[:,1]; pred_test=(p_test>=threshold).astype(int)
from sklearn.metrics import confusion_matrix, classification_report
metrics={"model":winner_name,"threshold":threshold,"roc_auc":float(roc_auc_score(y_test,p_test)),"pr_auc":float(average_precision_score(y_test,p_test)),"f1":float(f1_score(y_test,pred_test,zero_division=0)),"precision":float(precision_score(y_test,pred_test,zero_division=0)),"recall":float(recall_score(y_test,pred_test,zero_division=0)),"confusion_matrix":confusion_matrix(y_test,pred_test).tolist()}
print(json.dumps(metrics,indent=2))
fig,ax=plt.subplots(figsize=(5,4)); ConfusionMatrixDisplay.from_predictions(y_test,pred_test,ax=ax); ax.set_title("Held-out test confusion matrix"); plt.show()
fig,ax=plt.subplots(figsize=(6,4)); PrecisionRecallDisplay.from_predictions(y_test,p_test,ax=ax); ax.set_title("Held-out test precision-recall curve"); plt.show()
(ROOT/"reports"/"notebook_test_metrics.json").write_text(json.dumps(metrics,indent=2))

## Interpretation checklist
- Compare PR-AUC as well as ROC-AUC because churn is the minority class.
- Explain threshold choice in terms of outreach capacity and the cost of false negatives/false positives.
- Check the confusion matrix and class-specific precision/recall.
- Record package versions and the dataset source for reproducibility.
- For full experiment tracking, log model name, hyperparameters, validation metrics, threshold, plots and artifact path to MLflow; do not log raw customer data.